# Observation of High Intensity Radiation by Satellites 1958 Alpha and Gamma (Van Allen et al., 1958): Implementation / 구현

Van Allen 방사선대 발견 논문의 핵심 물리학을 직접 구현합니다: Geiger 관의 dead time 포화 메커니즘, 위성 데이터 재현, 방사선대 구조 시각화.

We implement the core physics of the Van Allen radiation belt discovery: Geiger tube dead time saturation mechanism, satellite data reproduction, and radiation belt structure visualization.

**구현 내용 / Implementation:**
1. Geiger 관 dead time 응답 함수 (Fig. 8 재현)
2. 우주선 강도 vs. 고도 (Fig. 3 재현)
3. 테이프 레코더 데이터 시뮬레이션 (Fig. 6 재현)
4. 궤도-방사선 강도 지도 (Fig. 7 재현)
5. 방사선량 계산 및 인체 영향 분석
6. Van Allen 방사선대 구조 시각화
7. 종합 — 현대 우주 공학적 함의

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Circle, FancyArrowPatch
from matplotlib.colors import LogNorm
import matplotlib.gridspec as gridspec

plt.rcParams.update({
    "figure.figsize": (10, 6),
    "font.size": 11,
    "axes.grid": True,
    "grid.alpha": 0.3,
    "axes.spines.top": False,
    "axes.spines.right": False,
})

# Physical constants
R_E = 6371.0  # Earth radius (km)
B0_eq = 3.1e-5  # Equatorial surface magnetic field (T)
k_B = 1.381e-23  # Boltzmann constant (J/K)
e_charge = 1.602e-19  # Elementary charge (C)
m_proton = 1.673e-27  # Proton mass (kg)
m_electron = 9.109e-31  # Electron mass (kg)

print("Constants loaded / 상수 로드 완료")

## Part 1: Geiger 관 Dead Time 응답 함수 — Fig. 8 재현 / Geiger Tube Dead Time Response — Fig. 8 Reproduction

논문의 핵심 논증: Geiger 관에 방사선 강도가 증가하면 관측 계수율이 **비단조적(non-monotonic)**으로 변합니다.

The paper's core argument: as radiation intensity increases, the observed count rate changes **non-monotonically**.

표준 dead time 모델: $m = n / (1 + n\tau)$

그러나 실제로는 $n$이 매우 클 때 펄스 진폭이 감소하여 스케일링 회로 임계값 아래로 떨어집니다. 이 효과를 포함한 확장 모델을 구현합니다.

Standard dead time model: $m = n / (1 + n\tau)$

In reality, at very large $n$, pulse amplitudes decrease below the scaling circuit threshold. We implement an extended model including this effect.

In [ ]:
def gm_response_standard(n: np.ndarray, tau: float = 100e-6) -> np.ndarray:
    """Standard non-paralyzable dead time model.

    Args:
        n: True count rate (counts/sec).
        tau: Dead time (sec), default 100 microseconds.

    Returns:
        Observed count rate m = n / (1 + n*tau).
    """
    return n / (1.0 + n * tau)


def gm_response_realistic(
    n: np.ndarray,
    tau: float = 100e-6,
    n_onset: float = 2000.0,
    n_blanking: float = 35000.0,
) -> np.ndarray:
    """Realistic G.M. tube response including pulse amplitude reduction.

    At high rates, continuous discharges reduce pulse amplitudes below
    the scaling circuit threshold. Models this as a smooth suppression
    factor multiplied onto the standard dead time formula.

    Args:
        n: True count rate (counts/sec).
        tau: Dead time (sec).
        n_onset: Rate at which amplitude reduction begins.
        n_blanking: Rate at which all pulses fall below threshold (m -> 0).

    Returns:
        Effective observed count rate.
    """
    m_standard = gm_response_standard(n, tau)

    # Amplitude suppression factor: smooth transition using a sigmoid
    # Below n_onset: factor ~ 1 (no suppression)
    # Above n_blanking: factor ~ 0 (complete blanking)
    n_mid = (n_onset + n_blanking) / 2
    steepness = 6.0 / (n_blanking - n_onset)  # ~99% transition in range
    suppression = 1.0 / (1.0 + np.exp(steepness * (n - n_mid)))

    return m_standard * suppression


# Compute response curves
n_true = np.logspace(0, 5.5, 1000)  # 1 to ~300,000 counts/sec

m_standard = gm_response_standard(n_true)
m_realistic = gm_response_realistic(n_true)

# --- Plot: Fig. 8 reproduction ---
fig, ax = plt.subplots(figsize=(9, 7))

ax.loglog(n_true, n_true, "k--", alpha=0.4, label="Ideal (no dead time) / 이상적 (dead time 없음)")
ax.loglog(n_true, m_standard, "b-", lw=1.5, alpha=0.6,
          label=f"Standard dead time ($\\tau$ = {100} μs)")
ax.loglog(
    n_true, np.maximum(m_realistic, 0.1), "r-", lw=2.5,
    label="Realistic (dead time + amplitude reduction)\n현실적 (dead time + 진폭 감소)",
)

# Mark key regions
ax.axvline(35000, color="gray", ls=":", alpha=0.5)
ax.annotate(
    "Blanking threshold\n포화 임계값\n≥ 35,000/sec",
    xy=(35000, 1), fontsize=9,
    xytext=(80000, 10), arrowprops=dict(arrowstyle="->", color="gray"),
    color="gray",
)

# Telemetry range of Explorer I
ax.axhspan(0.14 * 32, 80 * 32, color="green", alpha=0.08)
ax.text(2, 300, "1958α telemetry range\n(4.5–2,560 cts/sec)", fontsize=8, color="green")

# Mark the peak of realistic response
peak_idx = np.argmax(m_realistic)
ax.plot(n_true[peak_idx], m_realistic[peak_idx], "ro", ms=8)
ax.annotate(
    f"Peak: {m_realistic[peak_idx]:.0f}/sec\nat n = {n_true[peak_idx]:.0f}/sec",
    xy=(n_true[peak_idx], m_realistic[peak_idx]),
    xytext=(30, 5000), fontsize=9,
    arrowprops=dict(arrowstyle="->", color="red"),
    color="red",
)

ax.set_xlabel("Counts/sec without dead time effects\nDead time 효과 없는 계수율", fontsize=11)
ax.set_ylabel("Counts/sec with dead time effects\nDead time 효과 있는 계수율", fontsize=11)
ax.set_title("Fig. 8 Reproduction: Geiger Tube Response Function\nFig. 8 재현: Geiger 관 응답 함수", fontsize=13)
ax.legend(loc="upper left", fontsize=9)
ax.set_xlim(1, 3e5)
ax.set_ylim(0.1, 5e4)

plt.tight_layout()
plt.show()

## Part 2: 우주선 강도 vs. 고도 — Fig. 3 재현 / Cosmic Ray Intensity vs. Altitude — Fig. 3 Reproduction

캘리포니아 상공에서 Explorer I이 측정한 계수율 vs. 고도 데이터를 재현합니다. 고도 1,000 km 이하에서 계수율이 고도에 따라 **단조 증가**하며, 100 km로 외삽하면 $J_{\text{omni}} \approx 1.22 \, (\text{cm}^2 \cdot \text{sec})^{-1}$과 일치합니다.

Reproducing Explorer I count rate vs. altitude data over California. Below 1,000 km, count rate **increases monotonically** with altitude, and extrapolation to 100 km yields $J_{\text{omni}} \approx 1.22 \, (\text{cm}^2 \cdot \text{sec})^{-1}$.

이 증가의 물리적 원인: 대기에 의한 우주선 흡수가 감소하기 때문입니다 (대기 깊이가 줄어듦).

Physical cause of this increase: decreasing cosmic ray absorption by the atmosphere (reduced atmospheric depth).

In [ ]:
# Simulated data points from Fig. 3 (digitized approximations)
# Heights (km) and counting rates (counts/sec) from JPL California stations
heights_fig3 = np.array([
    250, 300, 340, 380, 400, 420, 450, 480, 500, 520, 550,
    580, 620, 650, 700, 750, 800, 850, 900, 950, 1000,
    1050, 1100, 1200, 1300, 1400, 1500,
])
# Add scatter to simulate real data (latitude variation + orbital uncertainty)
np.random.seed(42)
base_rate = 18 + 0.048 * heights_fig3  # Linear trend from paper
scatter = np.random.normal(0, 4, len(heights_fig3))
rates_fig3 = base_rate + scatter
rates_fig3 = np.clip(rates_fig3, 5, None)

# Linear fit for extrapolation
from numpy.polynomial.polynomial import polyfit
coeffs = polyfit(heights_fig3[heights_fig3 <= 1100], rates_fig3[heights_fig3 <= 1100], 1)
h_extrap = np.linspace(0, 1600, 200)
rate_extrap = coeffs[0] + coeffs[1] * h_extrap

# Omnidirectional intensity conversion
# G.M. tube: length 4 in = 10.16 cm, diameter 0.781 in = 1.98 cm
# Geometric factor G ≈ pi * r * L ≈ pi * 0.99 * 10.16 ≈ 31.6 cm^2 sr
# With 85% efficiency: effective area ~ 26.9 cm^2
# At 100 km: rate ≈ coeffs[0] + coeffs[1]*100
rate_100km = coeffs[0] + coeffs[1] * 100
G_eff = 26.9  # Effective geometric factor (cm^2)
J_omni_100 = rate_100km / G_eff

# Also show what happens above 1100 km (radiation belt onset)
h_belt = np.array([1100, 1200, 1300, 1400, 1500])
# True rate rises exponentially, but observed rate drops to zero
true_rate_belt = 50 * np.exp(0.005 * (h_belt - 1100))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

# Left: Fig. 3 reproduction
ax1.scatter(heights_fig3[heights_fig3 <= 1100], rates_fig3[heights_fig3 <= 1100],
            c="steelblue", s=40, zorder=5, label="Normal region / 정상 영역")
ax1.scatter(heights_fig3[heights_fig3 > 1100], rates_fig3[heights_fig3 > 1100],
            c="red", s=60, marker="^", zorder=5, label="Anomalous (entering belt) / 이상 영역")
ax1.plot(h_extrap[h_extrap <= 1200], rate_extrap[h_extrap <= 1200],
         "k--", alpha=0.5, label="Linear fit / 선형 적합")
ax1.axhline(rate_100km, color="green", ls=":", alpha=0.5)
ax1.text(50, rate_100km + 2, f"J(100 km) ≈ {J_omni_100:.2f} (cm²·s)⁻¹", fontsize=9, color="green")

# Mark radiation belt onset
ax1.axvspan(1100, 1600, color="red", alpha=0.08)
ax1.text(1150, 15, "Radiation belt\nonset / 방사선대\n시작", fontsize=8, color="red")

ax1.set_xlabel("Height / 고도 (km)")
ax1.set_ylabel("Counting Rate / 계수율 (sec⁻¹)")
ax1.set_title("Fig. 3: Count Rate vs. Height (California)\n계수율 vs. 고도 (캘리포니아)")
ax1.legend(fontsize=9)
ax1.set_xlim(0, 1600)
ax1.set_ylim(0, 140)

# Right: What the detector WOULD have seen vs. what it DID see
h_full = np.linspace(200, 2500, 500)
# Model: cosmic ray baseline + radiation belt contribution
cosmic_ray_baseline = coeffs[0] + coeffs[1] * h_full
# Radiation belt: exponential onset above ~1000 km
belt_contribution = np.where(
    h_full > 1000,
    500 * np.exp(0.004 * (h_full - 1000)) - 500,
    0,
)
true_total = cosmic_ray_baseline + belt_contribution
observed_total = gm_response_realistic(true_total)

ax2.semilogy(h_full, true_total, "r-", lw=2, label="True rate / 실제 계수율")
ax2.semilogy(h_full, np.maximum(observed_total, 0.1), "b-", lw=2,
             label="Observed (with dead time) / 관측 (dead time 포함)")
ax2.axhline(50, color="gray", ls=":", alpha=0.5)
ax2.text(300, 60, "Normal cosmic ray rate ~50/sec", fontsize=8, color="gray")
ax2.axvline(1100, color="orange", ls="--", alpha=0.5)
ax2.text(1150, 3, "Belt onset\n~1100 km", fontsize=8, color="orange")

ax2.set_xlabel("Height / 고도 (km)")
ax2.set_ylabel("Counting Rate / 계수율 (sec⁻¹)")
ax2.set_title("True vs. Observed Rate: The Paradox\n실제 vs. 관측 계수율: 역설")
ax2.legend(fontsize=9)
ax2.set_xlim(200, 2500)
ax2.set_ylim(0.1, 1e6)

plt.tight_layout()
plt.show()

print(f"Linear extrapolation to 100 km: {rate_100km:.1f} counts/sec")
print(f"Omnidirectional intensity at 100 km: {J_omni_100:.2f} (cm²·sec)⁻¹")
print(f"Paper value: 1.22 (cm²·sec)⁻¹")

## Part 3: 테이프 레코더 데이터 시뮬레이션 — Fig. 6 재현 / Tape Recorder Data Simulation — Fig. 6 Reproduction

Explorer III (1958γ)의 자기 테이프 레코더가 기록한 전 궤도 데이터를 시뮬레이션합니다. 3월 28일 San Diego 근처 readout (1748 UT)을 재현합니다.

Simulating the full-orbit data recorded by Explorer III's magnetic tape recorder. Reproducing the March 28 readout near San Diego (1748 UT).

핵심 관측: 패스 양 끝(저고도)에서는 정상 계수율, 중간(고고도)에서는 15분간 **0 계수율** — blanking의 직접적 증거.

Key observation: normal count rates at both ends (low altitude), **zero count rate** for 15 minutes in the middle (high altitude) — direct evidence of blanking.

In [ ]:
def simulate_explorer3_pass(
    total_time: float = 120.0,
    perigee_alt: float = 190.0,
    apogee_alt: float = 2500.0,
) -> tuple:
    """Simulate one orbital pass of Explorer III through the radiation belt.

    Args:
        total_time: Total pass duration (minutes).
        perigee_alt: Perigee altitude (km).
        apogee_alt: Apogee altitude (km).

    Returns:
        Tuple of (time_min, altitude_km, true_rate, observed_rate).
    """
    t = np.linspace(0, total_time, 1000)

    # Simplified sinusoidal altitude profile
    # Perigee at t=0 and t=total_time, apogee at t=total_time/2
    alt = perigee_alt + (apogee_alt - perigee_alt) * np.sin(np.pi * t / total_time) ** 2

    # True radiation rate model:
    # - Cosmic ray baseline: ~30-50/sec, slowly increasing with altitude
    # - Radiation belt: exponential onset above ~800 km, peaks at ~2000 km
    cosmic_ray = 30 + 0.02 * alt
    belt = np.where(
        alt > 800,
        200 * np.exp(0.003 * (alt - 800)),
        0,
    )
    true_rate = cosmic_ray + belt

    # Observed rate through G.M. tube response
    obs_rate = gm_response_realistic(true_rate)

    return t, alt, true_rate, obs_rate


t_min, alt_km, true_rate, obs_rate = simulate_explorer3_pass()

# What the tape recorder actually shows: scale of 128
# Max 1 scaler output per second → max displayed = 128 counts/sec
tape_display = np.clip(obs_rate, 0, 128)

fig, axes = plt.subplots(3, 1, figsize=(12, 10), sharex=True)

# Panel 1: Altitude profile
axes[0].plot(t_min, alt_km, "k-", lw=2)
axes[0].axhline(1100, color="red", ls="--", alpha=0.5)
axes[0].text(2, 1200, "Belt onset ~1100 km", fontsize=9, color="red")
axes[0].set_ylabel("Altitude / 고도 (km)")
axes[0].set_title("Fig. 6 Reproduction: Explorer III Tape Recorder Readout (March 28, 1748 UT)\n"
                   "Fig. 6 재현: Explorer III 테이프 레코더 데이터 (3월 28일, 1748 UT)")
axes[0].fill_between(t_min, 0, alt_km, where=alt_km > 1100, color="red", alpha=0.1)

# Panel 2: True rate (what a perfect detector would see)
axes[1].semilogy(t_min, true_rate, "r-", lw=2, label="True rate / 실제 계수율")
axes[1].axhline(35000, color="gray", ls=":", alpha=0.5)
axes[1].text(62, 45000, "Blanking threshold (35,000/sec)", fontsize=8, color="gray")
axes[1].set_ylabel("True Rate / 실제 계수율 (sec⁻¹)")
axes[1].legend(fontsize=9)

# Panel 3: What the tape recorder shows (as in Fig. 6)
axes[2].plot(t_min, tape_display, "b-", lw=2, label="Tape recorder output / 테이프 출력")
axes[2].axhline(128, color="orange", ls=":", alpha=0.5)
axes[2].text(2, 132, "Max recordable: 128/sec", fontsize=8, color="orange")

# Highlight the "zero" region
zero_mask = tape_display < 1
if np.any(zero_mask):
    zero_start = t_min[zero_mask][0]
    zero_end = t_min[zero_mask][-1]
    axes[2].axvspan(zero_start, zero_end, color="red", alpha=0.15)
    duration = zero_end - zero_start
    axes[2].annotate(
        f"\"Zero\" region: {duration:.0f} min\n(Paper: ~15 min)\nActual rate: >35,000/sec!",
        xy=((zero_start + zero_end) / 2, 0),
        xytext=(80, 80), fontsize=9, color="red",
        arrowprops=dict(arrowstyle="->", color="red"),
    )

axes[2].set_ylabel("Recorded Rate / 기록 계수율 (sec⁻¹)")
axes[2].set_xlabel("Time from previous interrogation / 이전 질의 후 시간 (minutes)")
axes[2].legend(fontsize=9)
axes[2].set_ylim(-5, 145)

plt.tight_layout()
plt.show()

# Calculate: total counts in zero region (as paper reports)
if np.any(zero_mask):
    print(f"Zero region duration: {duration:.1f} min")
    print(f"Total counts in zero region: ~128 (as paper reports)")
    print(f"Average rate: 128 / ({duration:.0f} × 60) = {128 / (duration * 60):.3f}/sec")
    print(f"Normal cosmic ray rate: ~50/sec")

## Part 4: 궤도-방사선 강도 지도 — Fig. 7 재현 / Orbit-Radiation Intensity Map — Fig. 7 Reproduction

3월 28~31일 Explorer III의 여러 궤도를 위도-경도 평면에 표시하고, 계수율 범위를 색상으로 구분합니다. 이 그림이 방사선대의 **공간적 구조**를 처음으로 보여준 증거입니다.

Plotting multiple Explorer III orbits from March 28–31 on the latitude-longitude plane, color-coded by count rate range. This figure was the first evidence of the **spatial structure** of the radiation belt.

핵심 패턴: 저위도(= 고고도) + 특정 경도 범위에서 고강도 방사선 집중.

Key pattern: high-intensity radiation concentrated at low latitudes (= high altitudes) + certain longitude ranges.

In [ ]:
def explorer3_ground_track(
    orbit_num: int,
    n_points: int = 500,
    inclination: float = 33.3,
    period_min: float = 115.7,
    perigee_lat: float = 33.0,
) -> tuple:
    """Generate a simplified Explorer III ground track.

    Args:
        orbit_num: Orbit number (shifts longitude due to Earth rotation).
        n_points: Number of track points.
        inclination: Orbital inclination (degrees).
        period_min: Orbital period (minutes).
        perigee_lat: Latitude of perigee (degrees).

    Returns:
        Tuple of (latitudes, longitudes, altitudes) arrays.
    """
    # Simplified orbital mechanics for visualization
    t = np.linspace(0, 1, n_points)  # Fraction of orbit

    # Latitude oscillates with inclination
    lat = inclination * np.sin(2 * np.pi * t)

    # Longitude drifts westward due to Earth rotation (~0.25 deg/min)
    earth_rotation_per_orbit = 0.25 * period_min  # degrees
    base_lon = 280 - orbit_num * earth_rotation_per_orbit
    lon = base_lon - 360 * t
    lon = ((lon + 180) % 360) - 180  # Wrap to [-180, 180]

    # Altitude: perigee near highest latitude, apogee near equator
    perigee_alt = 190.0
    apogee_alt = 2500.0
    alt = perigee_alt + (apogee_alt - perigee_alt) * np.cos(2 * np.pi * t) ** 2
    # Shift so perigee is at highest latitude
    alt = perigee_alt + (apogee_alt - perigee_alt) * (1 - np.cos(2 * np.pi * t) ** 2)

    return lat, lon, alt


fig, ax = plt.subplots(figsize=(14, 8))

# Generate multiple orbits (March 28-31)
colors_map = {
    "high": "red",      # >15,000/sec (blanked)
    "medium": "orange",  # 128-15,000/sec
    "low": "blue",       # <128/sec
}

for orbit_num in range(9):  # 9 orbits as in the paper
    lat, lon, alt = explorer3_ground_track(orbit_num)

    # Classify each point by count rate range based on altitude
    for i in range(len(lat) - 1):
        if alt[i] > 1800:  # Deep in belt → blanked
            color = colors_map["high"]
            ls = "-"
        elif alt[i] > 1000:  # Transitional
            color = colors_map["medium"]
            ls = "--"
        else:  # Below belt
            color = colors_map["low"]
            ls = "-"

        # Only plot if consecutive points are close in longitude
        if abs(lon[i + 1] - lon[i]) < 50:
            ax.plot([lon[i], lon[i + 1]], [lat[i], lat[i + 1]],
                    color=color, ls=ls, lw=1.5, alpha=0.7)

# Legend
from matplotlib.lines import Line2D
legend_elements = [
    Line2D([0], [0], color="red", lw=2, label="> 15,000/sec (blanked / 포화)"),
    Line2D([0], [0], color="orange", lw=2, ls="--",
           label="128–15,000/sec (transition / 전이)"),
    Line2D([0], [0], color="blue", lw=2, label="< 128/sec (normal / 정상)"),
]
ax.legend(handles=legend_elements, loc="lower left", fontsize=10)

# Mark ground stations from the paper
stations = {
    "San Diego": (-117.2, 32.7),
    "Quito": (-78.5, -0.2),
    "Antofagasta": (-70.4, -23.6),
    "Singapore": (103.8, 1.4),
    "Ibadan": (3.9, 7.4),
}
for name, (slon, slat) in stations.items():
    ax.plot(slon, slat, "k^", ms=8, zorder=10)
    ax.text(slon + 3, slat + 1.5, name, fontsize=7, color="black")

ax.set_xlabel("Geographic Longitude / 지리적 경도 (°)")
ax.set_ylabel("Geographic Latitude / 지리적 위도 (°)")
ax.set_title("Fig. 7 Reproduction: Explorer III Orbits with Count Rate Ranges\n"
             "Fig. 7 재현: Explorer III 궤도 및 계수율 범위 (March 28–31)")
ax.set_xlim(-180, 180)
ax.set_ylim(-40, 40)
ax.axhline(0, color="gray", ls="-", alpha=0.2)

plt.tight_layout()
plt.show()

## Part 5: 방사선량 계산 및 인체 영향 / Radiation Dose Calculations and Human Impact

논문에서 보고한 60 mR/hr의 의미를 현대 방사선 방호 기준과 비교하고, 방사선대 내 체류 시간에 따른 누적 선량을 계산합니다.

Comparing the paper's reported 60 mR/hr with modern radiation protection standards and calculating cumulative dose as a function of residence time in the radiation belt.

단위 변환: $1 \, \text{R} \approx 1 \, \text{rad} \approx 0.01 \, \text{Gy}$ (연조직), $1 \, \text{R} \approx 1 \, \text{rem} \approx 0.01 \, \text{Sv}$ (X선/감마선)

Unit conversion: $1 \, \text{R} \approx 1 \, \text{rad} \approx 0.01 \, \text{Gy}$ (soft tissue), $1 \, \text{R} \approx 1 \, \text{rem} \approx 0.01 \, \text{Sv}$ (X-rays/gamma rays)

In [ ]:
# Radiation dose calculations from the paper
dose_rate_blanking_mR_hr = 60.0  # mR/hr at blanking threshold
dose_rate_blanking_R_hr = dose_rate_blanking_mR_hr / 1000.0  # R/hr
dose_rate_blanking_mSv_hr = dose_rate_blanking_R_hr * 10.0  # mSv/hr (1 R ≈ 10 mSv)

# 1958 permissible dose
permissible_1958_R_week = 0.3  # R/week

# Modern limits (ICRP)
occupational_annual_mSv = 50.0  # mSv/year
public_annual_mSv = 1.0  # mSv/year
iss_6month_mSv = 120.0  # typical ISS 6-month dose
acute_threshold_mSv = 500.0  # acute radiation syndrome

# Time to reach various limits at blanking dose rate
time_to_1958_limit_hr = permissible_1958_R_week / dose_rate_blanking_R_hr
time_to_public_limit_hr = public_annual_mSv / dose_rate_blanking_mSv_hr
time_to_occupational_limit_hr = occupational_annual_mSv / dose_rate_blanking_mSv_hr
time_to_acute_hr = acute_threshold_mSv / dose_rate_blanking_mSv_hr

# --- Plot: Cumulative dose over time ---
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

# Left: Cumulative dose at different dose rates
hours = np.linspace(0, 100, 500)

# Multiple dose rate scenarios (the paper's 60 mR/hr is MINIMUM)
dose_rates_mSv_hr = [0.6, 3.0, 15.0, 100.0]  # mSv/hr
labels = [
    "60 mR/hr (paper minimum)\n60 mR/hr (논문 최소값)",
    "~300 mR/hr (moderate belt)\n~300 mR/hr (중간 방사선대)",
    "~1.5 R/hr (inner belt core)\n~1.5 R/hr (내대 중심부)",
    "~10 R/hr (storm-enhanced)\n~10 R/hr (폭풍 강화시)",
]
colors = ["blue", "orange", "red", "darkred"]

for rate, label, color in zip(dose_rates_mSv_hr, labels, colors):
    cumulative = rate * hours
    ax1.plot(hours, cumulative, color=color, lw=2, label=label)

# Reference lines
ax1.axhline(public_annual_mSv, color="green", ls=":", alpha=0.7)
ax1.text(70, public_annual_mSv + 20, "Public annual limit\n일반인 연간 한도 (1 mSv)",
         fontsize=7, color="green")
ax1.axhline(occupational_annual_mSv, color="purple", ls=":", alpha=0.7)
ax1.text(70, occupational_annual_mSv + 30, "Occupational limit\n직업 한도 (50 mSv/yr)",
         fontsize=7, color="purple")
ax1.axhline(acute_threshold_mSv, color="red", ls=":", alpha=0.7)
ax1.text(70, acute_threshold_mSv + 30, "Acute syndrome\n급성 방사선 증후군 (500 mSv)",
         fontsize=7, color="red")

ax1.set_xlabel("Time in belt / 방사선대 체류 시간 (hours)")
ax1.set_ylabel("Cumulative dose / 누적 선량 (mSv)")
ax1.set_title("Cumulative Radiation Dose\n누적 방사선량")
ax1.legend(fontsize=7, loc="upper left")
ax1.set_xlim(0, 100)
ax1.set_ylim(0, 600)

# Right: Comparison bar chart of dose rates
scenarios = [
    "Natural background\n자연 배경 방사선",
    "Paper minimum\n논문 최소값 (60 mR/hr)",
    "Inner belt (protons)\n내대 (양성자)",
    "Outer belt (storm)\n외대 (폭풍시)",
    "Chest X-ray\n흉부 X선 (1회)",
]
# Dose rates in mSv/hr (for comparison purposes)
dose_values = [
    0.3e-3,    # ~2.5 mSv/yr natural background
    0.6,       # Paper's 60 mR/hr
    50.0,      # Inner belt core
    500.0,     # Outer belt during storm
    0.02/0.001,  # Chest X-ray ~0.02 mSv in ~1 sec exposure → rate equivalent
]
# Actually, let's use total dose for intuitive comparison
scenario_labels = [
    "Natural\n(1 year)\n자연 (1년)",
    "Paper min.\n(1 hour)\n논문 최소\n(1시간)",
    "Inner belt\n(1 hour)\n내대 (1시간)",
    "ISS\n(6 months)\nISS (6개월)",
    "Chest X-ray\n흉부 X선",
]
dose_mSv = [2.5, 0.6, 50, 120, 0.02]
bar_colors = ["green", "blue", "red", "orange", "lightblue"]

bars = ax2.barh(scenario_labels, dose_mSv, color=bar_colors, edgecolor="gray", alpha=0.8)
ax2.set_xscale("log")
ax2.set_xlabel("Dose / 선량 (mSv)")
ax2.set_title("Radiation Dose Comparison\n방사선량 비교")
ax2.axvline(1, color="green", ls="--", alpha=0.5)
ax2.text(1.1, 4.3, "Public limit\n(1 mSv/yr)", fontsize=7, color="green")
ax2.axvline(500, color="red", ls="--", alpha=0.5)
ax2.text(550, 4.3, "Acute syndrome\n(500 mSv)", fontsize=7, color="red")

for bar, val in zip(bars, dose_mSv):
    ax2.text(val * 1.3, bar.get_y() + bar.get_height() / 2,
             f"{val} mSv", va="center", fontsize=8)

plt.tight_layout()
plt.show()

print("=" * 60)
print("At paper's MINIMUM dose rate (60 mR/hr = 0.6 mSv/hr):")
print(f"  Time to 1958 weekly limit (0.3 R): {time_to_1958_limit_hr:.1f} hours")
print(f"  Time to modern public annual limit (1 mSv): {time_to_public_limit_hr:.1f} hours")
print(f"  Time to occupational annual limit (50 mSv): {time_to_occupational_limit_hr:.1f} hours")
print(f"  Time to acute syndrome (500 mSv): {time_to_acute_hr:.1f} hours")
print("=" * 60)
print("NOTE: 60 mR/hr is the MINIMUM (at blanking threshold).")
print("Actual belt dose rates can be 10-1000x higher!")

## Part 6: Van Allen 방사선대 구조 시각화 / Van Allen Radiation Belt Structure Visualization

후속 연구에서 밝혀진 Van Allen 방사선대의 2중 구조를 시각화합니다. 쌍극자 자기장에서의 입자 갇힘과 내대/외대의 공간 분포를 보여줍니다.

Visualizing the dual structure of the Van Allen radiation belts revealed by subsequent research. Showing particle trapping in the dipole field and the spatial distribution of inner/outer belts.

자기장: $B(r, \theta) = B_0 (R_E/r)^3 \sqrt{1 + 3\sin^2\lambda}$ (쌍극자), 여기서 $\lambda$는 자기 위도.

Magnetic field: $B(r, \theta) = B_0 (R_E/r)^3 \sqrt{1 + 3\sin^2\lambda}$ (dipole), where $\lambda$ is magnetic latitude.

In [ ]:
def dipole_field_line(L: float, n_points: int = 200) -> tuple:
    """Compute a dipole magnetic field line.

    Args:
        L: L-shell parameter (equatorial distance in Earth radii).
        n_points: Number of points.

    Returns:
        Tuple of (x, y) coordinates in Earth radii.
    """
    theta = np.linspace(-np.pi / 2, np.pi / 2, n_points)
    r = L * np.cos(theta) ** 2  # Dipole field line equation
    x = r * np.cos(theta)
    y = r * np.sin(theta)
    return x, y


def radiation_belt_intensity(r: np.ndarray, theta: np.ndarray) -> np.ndarray:
    """Model radiation belt particle intensity as function of position.

    Simplified model with inner belt (L~1.5-2.5) and outer belt (L~3.5-6).

    Args:
        r: Radial distance in Earth radii.
        theta: Magnetic colatitude (radians from equatorial plane).

    Returns:
        Normalized intensity (0-1).
    """
    # L-shell at each point (approximate)
    cos_lat = np.cos(theta)
    L = np.where(cos_lat > 0.01, r / (cos_lat ** 2), 100)

    # Inner belt: centered at L=1.8, width ~0.8
    inner = np.exp(-0.5 * ((L - 1.8) / 0.4) ** 2)

    # Outer belt: centered at L=4.5, width ~1.5
    outer = 0.7 * np.exp(-0.5 * ((L - 4.5) / 1.0) ** 2)

    # Latitude dependence: particles concentrated near equator
    lat_factor = np.exp(-2 * theta ** 2)

    intensity = (inner + outer) * lat_factor
    return np.clip(intensity, 0, 1)


fig, ax = plt.subplots(figsize=(12, 8))

# Create radiation belt intensity map
x_grid = np.linspace(-7, 7, 500)
y_grid = np.linspace(-4, 4, 300)
X, Y = np.meshgrid(x_grid, y_grid)
R = np.sqrt(X**2 + Y**2)
THETA = np.arctan2(Y, X)  # Angle from equatorial plane (x-axis)

# Mask inside Earth
mask = R < 1.0
intensity = radiation_belt_intensity(R, THETA)
intensity[mask] = np.nan

# Plot intensity
im = ax.pcolormesh(X, Y, intensity, cmap="hot_r", shading="auto",
                   vmin=0, vmax=1, alpha=0.8)
plt.colorbar(im, ax=ax, label="Relative Particle Intensity / 상대 입자 강도", shrink=0.8)

# Draw dipole field lines
for L in [1.5, 2.0, 2.5, 3.0, 4.0, 5.0, 6.0]:
    fx, fy = dipole_field_line(L)
    ax.plot(fx, fy, "gray", lw=0.5, alpha=0.5)

# Draw Earth
earth = Circle((0, 0), 1.0, facecolor="steelblue", edgecolor="black", lw=2, zorder=10)
ax.add_patch(earth)
ax.text(0, 0, "Earth", ha="center", va="center", fontsize=9, fontweight="bold",
        color="white", zorder=11)

# Explorer I/III orbit (approximate)
orbit_theta = np.linspace(0, 2 * np.pi, 200)
orbit_perigee = 1 + 190 / R_E  # ~1.03 R_E
orbit_apogee = 1 + 2500 / R_E   # ~1.39 R_E
orbit_a = (orbit_perigee + orbit_apogee) / 2
orbit_e = (orbit_apogee - orbit_perigee) / (orbit_apogee + orbit_perigee)
orbit_r = orbit_a * (1 - orbit_e ** 2) / (1 + orbit_e * np.cos(orbit_theta))
orbit_x = orbit_r * np.cos(orbit_theta)
orbit_y = orbit_r * np.sin(orbit_theta)
ax.plot(orbit_x, orbit_y, "cyan", lw=2, ls="--", label="Explorer I/III orbit\nExplorer I/III 궤도", zorder=5)

# Annotate belt regions
ax.annotate("Inner Belt\n내대\n(protons)\nL ~ 1.5–2.5",
            xy=(1.8, 0), xytext=(1.8, 2.5),
            fontsize=9, ha="center", color="darkred",
            arrowprops=dict(arrowstyle="->", color="darkred"))
ax.annotate("Outer Belt\n외대\n(electrons)\nL ~ 3.5–6",
            xy=(4.5, 0), xytext=(4.5, 2.5),
            fontsize=9, ha="center", color="darkred",
            arrowprops=dict(arrowstyle="->", color="darkred"))
ax.annotate("Slot Region\n슬롯 영역\nL ~ 2.5–3.5",
            xy=(3.0, 0), xytext=(3.0, -2.5),
            fontsize=9, ha="center", color="navy",
            arrowprops=dict(arrowstyle="->", color="navy"))

# Mark Baker (2014) impenetrable barrier
barrier_L = 2.8
bx, by = dipole_field_line(barrier_L)
ax.plot(bx, by, "lime", lw=2, ls="-.", alpha=0.8,
        label=f"\"Impenetrable barrier\" (L={barrier_L})\n\"뚫을 수 없는 장벽\" (Baker 2014)")

ax.set_xlabel("Distance / 거리 ($R_E$)")
ax.set_ylabel("Distance / 거리 ($R_E$)")
ax.set_title("Van Allen Radiation Belts: Structure & Explorer I/III Orbit\n"
             "Van Allen 방사선대: 구조 및 Explorer I/III 궤도")
ax.set_xlim(-7, 7)
ax.set_ylim(-4, 4)
ax.set_aspect("equal")
ax.legend(loc="lower left", fontsize=9)

plt.tight_layout()
plt.show()

print(f"Explorer I/III perigee: {190:.0f} km ({orbit_perigee:.2f} R_E)")
print(f"Explorer I/III apogee: {2500:.0f} km ({orbit_apogee:.2f} R_E)")
print(f"Inner belt range: L = 1.5–2.5 R_E")
print(f"Orbit penetrates lower edge of inner belt")

## Part 7: 종합 — 현대 우주 공학적 함의 / Synthesis — Modern Space Engineering Implications

Van Allen의 발견이 현대 위성 설계와 유인 우주비행에 미친 영향을 정량적으로 분석합니다. 방사선대를 통과하는 다양한 궤도에서의 선량 적분, Apollo 미션의 방사선대 통과 전략, ISS 궤도가 방사선대 아래인 이유를 보여줍니다.

Quantitatively analyzing the impact of Van Allen's discovery on modern satellite design and human spaceflight. Showing dose integration for various orbits through the belts, Apollo's belt traversal strategy, and why the ISS orbits below the belts.

| 논문 연결 / Paper Connection | 현대 응용 / Modern Application |
|---|---|
| #2 Chapman-Ferraro → 자기권 | 자기권 모델 → 방사선대 위치 예측 / Magnetosphere models → belt location prediction |
| #4 Parker → 태양풍 | 태양풍 변동 → 방사선대 강도 변화 / Solar wind variation → belt intensity changes |
| ★ Van Allen → 방사선대 | 위성 차폐, 궤도 선택, 유인비행 방호 / Satellite shielding, orbit selection, human spaceflight protection |

In [ ]:
def belt_dose_rate_model(altitude_km: np.ndarray) -> np.ndarray:
    """Simplified dose rate model as function of altitude (equatorial crossing).

    Args:
        altitude_km: Altitude above Earth's surface (km).

    Returns:
        Dose rate (mSv/hr).
    """
    r_Re = 1 + altitude_km / R_E  # Distance in Earth radii

    # Inner belt: peaks at ~1.5 R_E (~3200 km), mainly protons
    inner = 200 * np.exp(-0.5 * ((r_Re - 1.5) / 0.3) ** 2)

    # Outer belt: peaks at ~4.5 R_E (~22300 km), mainly electrons
    outer = 50 * np.exp(-0.5 * ((r_Re - 4.5) / 1.0) ** 2)

    # Slot region: reduced but not zero
    slot = 2 * np.exp(-0.5 * ((r_Re - 3.0) / 0.3) ** 2)

    return inner + outer + slot + 0.001  # Small background


# Various orbital scenarios
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# (a) Dose rate vs altitude profile
alt_range = np.linspace(0, 40000, 2000)
dose_profile = belt_dose_rate_model(alt_range)

ax = axes[0, 0]
ax.semilogy(alt_range / 1000, dose_profile, "r-", lw=2)
# Mark key altitudes
key_alts = {
    "ISS (~400 km)": 400,
    "Explorer I/III\napogee (~2500 km)": 2500,
    "GPS (~20200 km)": 20200,
    "GEO (~35800 km)": 35800,
}
for label, alt in key_alts.items():
    rate = belt_dose_rate_model(np.array([alt]))[0]
    ax.plot(alt / 1000, rate, "ko", ms=6, zorder=5)
    ax.annotate(label, xy=(alt / 1000, rate),
                xytext=(alt / 1000 + 2, rate * 3),
                fontsize=7, arrowprops=dict(arrowstyle="->", color="gray"))

ax.axhline(0.6, color="blue", ls=":", alpha=0.5)
ax.text(32, 0.7, "Paper's 60 mR/hr", fontsize=7, color="blue")
ax.set_xlabel("Altitude / 고도 (×1000 km)")
ax.set_ylabel("Dose rate / 선량률 (mSv/hr)")
ax.set_title("(a) Radiation Dose Rate vs. Altitude\n고도별 방사선량률")

# (b) Apollo trajectory through belts
ax = axes[0, 1]
# Apollo traversed belts in ~30 min each way through a low-dose corridor
t_apollo = np.linspace(0, 3, 200)  # hours (outbound through belt)
# Simplified altitude profile: rapid transit through belts
alt_apollo = 400 + 35000 * (t_apollo / 3)  # Linear climb to GEO+
dose_rate_apollo = belt_dose_rate_model(alt_apollo)
cumulative_apollo = np.cumsum(dose_rate_apollo) * (t_apollo[1] - t_apollo[0])

ax.plot(t_apollo, cumulative_apollo, "navy", lw=2, label="Outbound / 왕복")
ax.axhline(50, color="purple", ls=":", alpha=0.5)
ax.text(0.2, 53, "Occupational limit (50 mSv/yr)", fontsize=7, color="purple")

# Typical Apollo total belt dose: ~2-5 mSv per transit
total_dose = cumulative_apollo[-1]
ax.annotate(f"Total transit dose: {total_dose:.1f} mSv\n총 통과 선량",
            xy=(3, total_dose), xytext=(1.5, total_dose + 20),
            fontsize=9, arrowprops=dict(arrowstyle="->", color="navy"))

ax.set_xlabel("Transit time / 통과 시간 (hours)")
ax.set_ylabel("Cumulative dose / 누적 선량 (mSv)")
ax.set_title("(b) Apollo Belt Transit Dose\nApollo 방사선대 통과 선량")
ax.legend(fontsize=9)

# (c) Why ISS orbits below the belts
ax = axes[1, 0]
orbit_alts = np.array([400, 1000, 2000, 3200, 5000, 10000, 20200, 35800])
orbit_names = ["ISS\n400 km", "LEO-high\n1000 km", "MEO-low\n2000 km",
               "Inner belt\npeak\n3200 km", "MEO\n5000 km",
               "MEO-high\n10000 km", "GPS\n20200 km", "GEO\n35800 km"]
yearly_doses = belt_dose_rate_model(orbit_alts) * 8760  # mSv per year

bar_colors_orbit = ["green" if d < 50 else "orange" if d < 500 else "red" for d in yearly_doses]
bars = ax.barh(range(len(orbit_alts)), yearly_doses, color=bar_colors_orbit,
               edgecolor="gray", alpha=0.8)
ax.set_yticks(range(len(orbit_alts)))
ax.set_yticklabels(orbit_names, fontsize=7)
ax.set_xscale("log")
ax.set_xlabel("Annual dose / 연간 선량 (mSv)")
ax.set_title("(c) Annual Dose by Orbit Altitude\n궤도 고도별 연간 선량")
ax.axvline(50, color="purple", ls="--", alpha=0.5)
ax.text(55, 7.3, "Occupational\nlimit", fontsize=7, color="purple")

# (d) Summary: Van Allen's discovery → modern implications
ax = axes[1, 1]
ax.axis("off")
summary_text = """
Van Allen (1958) → Modern Space Engineering
Van Allen (1958) → 현대 우주 공학

1. Satellite shielding / 위성 차폐
   All electronics require radiation hardening
   모든 전자장비에 방사선 경화 필요

2. Orbit selection / 궤도 선택
   ISS at 400 km: BELOW the belts
   GPS at 20,200 km: in the slot region
   GEO at 35,800 km: ABOVE inner belt

3. Human spaceflight / 유인 우주비행
   Apollo: rapid transit through belts (~30 min)
   Lunar Gateway: careful orbit design needed

4. Space weather forecasting / 우주기상 예보
   Belt intensity varies with solar activity
   Storm-time enhancement can be 10–100×

5. Next paper: Dungey (1961) / 다음 논문
   Magnetic reconnection explains HOW
   solar wind energy enters the belts
   자기 재결합이 태양풍 에너지가 방사선대에
   유입되는 메커니즘을 설명
"""
ax.text(0.05, 0.95, summary_text, transform=ax.transAxes,
        fontsize=9, verticalalignment="top", fontfamily="monospace",
        bbox=dict(boxstyle="round", facecolor="lightyellow", alpha=0.8))

plt.tight_layout()
plt.show()

## Summary / 요약

| Part | 구현 내용 / Implementation | Van Allen (1958)의 기여 / Contribution |
|------|--------------------------|---------------------------------------|
| 1 | Geiger 관 dead time 응답 함수 (Fig. 8) | 핵심 논증: 계수율 0 = 극도로 강한 방사선 / Core argument: zero rate = extremely intense radiation |
| 2 | 우주선 강도 vs. 고도 (Fig. 3) | 정상 작동 확인 + 이상 영역 식별 / Normal operation confirmed + anomalous region identified |
| 3 | 테이프 레코더 시뮬레이션 (Fig. 6) | 전 궤도 데이터의 결정적 역할 / Decisive role of full-orbit data |
| 4 | 궤도-계수율 지도 (Fig. 7) | 방사선대의 공간 구조 최초 매핑 / First spatial mapping of radiation belt structure |
| 5 | 방사선량 계산 | 인체 방사선 위험성 최초 정량화 / First quantification of human radiation hazard |
| 6 | 방사선대 구조 시각화 | 내대/외대/슬롯 구조와 Explorer 궤도 관계 / Inner/outer belt/slot structure and Explorer orbit relationship |
| 7 | 현대 우주 공학 함의 | 위성 설계, 궤도 선택, 유인비행 방호 / Satellite design, orbit selection, human spaceflight protection |

**다음 논문 / Next paper**: #6 Dungey (1961) — "Interplanetary Magnetic Field and the Auroral Zones" — 자기 재결합이 태양풍 에너지를 자기권/방사선대에 전달하는 메커니즘을 설명 / Explains how magnetic reconnection transfers solar wind energy to the magnetosphere and radiation belts.